In [29]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split

In [30]:
df = pd.read_csv('/kaggle/input/datasets/harshadapatil31/student-performance-and-study-habits-dataset/student_performance_dataset.csv')
df.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [31]:
df=df.drop('student_id',axis=True)

In [32]:
df['internet_access']=df['internet_access'].map({'Yes':1 ,'No':0})
df['extracurricular_activities']=df['extracurricular_activities'].map({'Yes':1 ,'No':0})
df['part_time_job']=df['part_time_job'].map({'Yes':1 ,'No':0})

In [33]:
df.isna().sum()

gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

In [34]:
df['parental_education']=df['parental_education'].fillna(df['parental_education'].mode()[0])

In [35]:
df.isna().sum()

gender                        0
study_time_hours              0
attendance_percent            0
sleep_hours                   0
parental_education            0
internet_access               0
extracurricular_activities    0
part_time_job                 0
previous_grade                0
final_exam_score              0
final_grade                   0
dtype: int64

In [36]:
lbl=LabelEncoder()
df['final_grade']=lbl.fit_transform(df['final_grade'])

In [37]:
df.head()

,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,Male,4.0,98.0,6.5,Bachelors,1,1,0,76.9,100.0,0
1,Female,6.3,100.0,5.7,High School,1,1,1,75.5,100.0,0
2,Male,4.9,85.3,7.9,Bachelors,1,0,1,88.5,97.3,0
3,Male,2.6,77.5,8.0,High School,1,1,0,85.1,83.8,1
4,Male,2.2,89.6,4.6,Bachelors,1,0,1,61.8,68.3,3


In [38]:
X=df.drop('final_grade',axis=1)
y=df['final_grade']

In [39]:
Categorical_columns = ['gender']
Ordinal_columns = ['parental_education']
numeric_columns = [feature for feature in X if X[feature].dtype=='int' or X[feature].dtype=='float']
print(numeric_columns)

['study_time_hours', 'attendance_percent', 'sleep_hours', 'internet_access', 'extracurricular_activities', 'part_time_job', 'previous_grade', 'final_exam_score']


In [40]:
ohe=OneHotEncoder()
oe=OrdinalEncoder(categories=[['High School','Bachelors','Masters','PhD']])
scaler=StandardScaler()

In [41]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [42]:
preprocessor = ColumnTransformer([
    ("OneHotEncoder",ohe,Categorical_columns),
    ("OrdinalEncoder",oe,Ordinal_columns),
    ("StandardScaler",scaler,numeric_columns)
],remainder='passthrough')

In [43]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [44]:
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train,
    columns=feature_names
)


In [45]:
X_train_df.head()

,OneHotEncoder__gender_Female,OneHotEncoder__gender_Male,OrdinalEncoder__parental_education,StandardScaler__study_time_hours,StandardScaler__attendance_percent,StandardScaler__sleep_hours,StandardScaler__internet_access,StandardScaler__extracurricular_activities,StandardScaler__part_time_job,StandardScaler__previous_grade,StandardScaler__final_exam_score
0,0.0,1.0,0.0,-0.614146,-1.280612,0.712201,0.410152,-1.191367,1.422742,-1.142853,-1.907532
1,1.0,0.0,1.0,0.997081,-1.419354,-0.981096,0.410152,0.839372,-0.702868,-1.487469,-0.331842
2,0.0,1.0,1.0,-0.345608,1.600957,-0.727102,0.410152,-1.191367,-0.702868,-1.703856,-0.815183
3,1.0,0.0,2.0,0.057199,-1.846253,-0.811767,0.410152,0.839372,1.422742,-1.687827,-1.201855
4,1.0,0.0,1.0,0.460005,0.085465,-0.473107,0.410152,0.839372,-0.702868,-0.028862,0.634839


In [46]:
class StudentDataset(Dataset):
    def __init__(self,X,y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        features = torch.tensor(self.X[idx],dtype=torch.float32)
        target = torch.tensor(self.y.values[idx],dtype=torch.long)
        return features,target

In [47]:
dataset = StudentDataset(X_train,y_train)
dataloader = DataLoader(dataset,batch_size = 32,shuffle = True,num_workers = 4)

In [48]:
for batch_idx,(features,targets) in enumerate(dataloader):
    print(f"Batch {batch_idx + 1}")
    print("Features : ",features.shape)
    print("Targets : ",targets.shape)
    break

Batch 1
Features :  torch.Size([32, 11])
Targets :  torch.Size([32])


In [49]:
class StudentPerformanceClassifier(nn.Module):
    def __init__(self,input_dim,output_dim):
        super(StudentPerformanceClassifier,self).__init__()

        self.network=nn.Sequential(
            nn.Linear(input_dim,128),
            nn.Dropout(p=0.2),
            nn.ReLU(),
            nn.Linear(128,32),
            nn.Dropout(p=0.2),
            nn.ReLU(),
            nn.Linear(32,output_dim),
            nn.Softmax()
        )

    def forward(self,x):
        return self.network(x)

In [50]:
input_dim = X_train.shape[1]
output_dim = 5


In [51]:
model = StudentPerformanceClassifier(input_dim,output_dim)

In [52]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

In [53]:
print(model)

StudentPerformanceClassifier(
  (network): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=32, bias=True)
    (4): Dropout(p=0.2, inplace=False)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=5, bias=True)
    (7): Softmax(dim=None)
  )
)


In [54]:
train_dataset = StudentDataset(X_train,y_train)
test_dataset = StudentDataset(X_test,y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [55]:
epochs=500
for epoch in range(epochs):

    
    model.train()
    train_correct = 0
    train_total = 0

    for batch_X, batch_y in train_loader:

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        predicted = outputs.argmax(dim=1)

        train_correct += (predicted == batch_y).sum().item()
        train_total += batch_y.size(0)

    train_acc = 100 * train_correct / train_total

    # Testing
    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for batch_X, batch_y in test_loader:

            outputs = model(batch_X)
            predicted = outputs.argmax(dim=1)

            test_correct += (predicted == batch_y).sum().item()
            test_total += batch_y.size(0)

    test_acc = 100 * test_correct / test_total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {loss.item():.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Test Acc: {test_acc:.2f}%"
    )

Epoch [1/500] Loss: 1.2315 Train Acc: 55.33% Test Acc: 74.80%
Epoch [2/500] Loss: 1.2014 Train Acc: 75.07% Test Acc: 80.80%


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch [3/500] Loss: 0.9899 Train Acc: 81.87% Test Acc: 78.40%
Epoch [4/500] Loss: 1.0022 Train Acc: 82.00% Test Acc: 82.00%
Epoch [5/500] Loss: 1.1144 Train Acc: 82.00% Test Acc: 80.80%
Epoch [6/500] Loss: 1.1158 Train Acc: 82.40% Test Acc: 79.60%
Epoch [7/500] Loss: 1.0195 Train Acc: 84.40% Test Acc: 81.20%
Epoch [8/500] Loss: 1.1381 Train Acc: 83.73% Test Acc: 82.40%
Epoch [9/500] Loss: 1.0839 Train Acc: 84.53% Test Acc: 81.60%
Epoch [10/500] Loss: 1.2629 Train Acc: 84.80% Test Acc: 79.60%
Epoch [11/500] Loss: 0.9908 Train Acc: 85.20% Test Acc: 82.80%
Epoch [12/500] Loss: 0.9955 Train Acc: 86.40% Test Acc: 80.00%
Epoch [13/500] Loss: 1.1014 Train Acc: 84.13% Test Acc: 81.20%
Epoch [14/500] Loss: 1.0466 Train Acc: 80.27% Test Acc: 79.20%
Epoch [15/500] Loss: 1.1871 Train Acc: 85.20% Test Acc: 79.20%
Epoch [16/500] Loss: 1.0157 Train Acc: 83.47% Test Acc: 76.80%
Epoch [17/500] Loss: 1.1191 Train Acc: 81.60% Test Acc: 80.00%
Epoch [18/500] Loss: 1.0485 Train Acc: 84.40% Test Acc: 81.20%

In [56]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:

        outputs = model(batch_X)
        predicted = outputs.argmax(dim=1)

        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

test_acc = 100 * correct / total

print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 78.00%
